# Classification NBA Model

## Configuration

## Imports

In [42]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from nba_ou.data_preparation.missing_data.clean_df_for_training import (
    clean_dataframe_for_training,
)
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
)
from sklearn.model_selection import cross_validate
from xgboost import XGBRegressor


In [43]:
from nba_ou.modeling.modeling import (
    TemporalDecaySampleWeightRegressor,
    assert_valid_time_splits,
    build_recency_sample_weights,
    evaluate_day_by_day_walk_forward,
    make_test_anchored_walk_forward_splits,
    save_model_bundle,
    load_model_bundle,
    split_latest_dates_holdout,
)


In [44]:
nan_threshold = 5
max_na_per_row = 5

## Load Data

In [45]:
exclude = "fanatics_sportsbook"

In [46]:
data_path = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/data/train_data/"
name = "all_odds_training_data_until_20260408.csv"

path = data_path + name

df_stats = pd.read_csv(path)

dtype_dict = {col: str for col in df_stats.columns if "ID" in col.upper()}

df_stats = pd.read_csv(
    path,
    dtype=dtype_dict
)
df_stats['GAME_DATE'] = pd.to_datetime(df_stats['GAME_DATE']).dt.strftime('%Y-%m-%d')

In [47]:
df_to_train = clean_dataframe_for_training(df_stats, nan_threshold=nan_threshold, max_na_per_row=max_na_per_row, create_missing_flags=False, verbose=1, keep_columns=['GAME_DATE'], exclude_cols_containing=[exclude])

STARTING DATAFRAME CLEANING PIPELINE
Starting basic cleaning with 11416 rows
Basic cleaning complete: 8807 rows remaining

Starting advanced column cleaning with 2948 columns

Advanced column cleaning complete: 2948 → 1373 columns (1575 removed)


Applying missing data policy...

Missing Data Policy Report:
  Rows dropped: 1 (0.01%)
  Critical columns requiring data: 4
  Columns zero-filled: 104
  Infer pairs applied: 20/136
  Remaining NaN cells: 135191

Dropping rows with more than 5 NaN values...
Removed 1414 rows exceeding NaN threshold
CLEANING COMPLETE
Final shape: (7392, 1373)


In [48]:
# Count NAs per column
na_counts = df_to_train.isna().sum()

# Get most common SEASON_YEAR for nulls in each column
most_common_season = []
for col in df_to_train.columns:
    if na_counts[col] > 0:
        # Get rows where this column is null
        null_rows = df_to_train[df_to_train[col].isna()]
        if len(null_rows) > 0 and 'SEASON_YEAR' in df_to_train.columns:
            # Find most common SEASON_YEAR for these null rows
            common_season = null_rows['SEASON_YEAR'].mode()
            most_common_season.append(common_season.iloc[0] if len(common_season) > 0 else None)
        else:
            most_common_season.append(None)
    else:
        most_common_season.append(None)

na_counts_df = pd.DataFrame({
    'Column': na_counts.index,
    'NA_Count': na_counts.values,
    'NA_Percentage': (na_counts.values / len(df_to_train) * 100).round(2),
    'Most_Common_Season_Year': most_common_season
}).sort_values('NA_Count', ascending=False)

# Show only columns with NAs
na_counts_df[na_counts_df['NA_Count'] > 0]

,Column,NA_Count,NA_Percentage,Most_Common_Season_Year
1171,LEAGUE_GAMES_LAST_1D_BEFORE,272,3.68,2023.0
1363,TRAVEL_RECENCY_RATIO_AWAY_2D_OVER_14D_BEFORE,79,1.07,2019.0
638,ml_betmgm_price_LAST_ALL_5_MATCHES_BEFORE_TEAM...,71,0.96,2019.0
219,ml_betmgm_price_LAST_ALL_5_MATCHES_BEFORE_TEAM...,71,0.96,2019.0
1362,TRAVEL_RECENCY_RATIO_HOME_2D_OVER_14D_BEFORE,41,0.55,2019.0
...,...,...,...,...
751,TOTAL_LINE_betmgm_SEASON_BEFORE_STD_TREND_SLOP...,1,0.01,2019.0
539,DIFF_FROM_LINE_caesars_LAST_ALL_2_MATCHES_BEFO...,1,0.01,2019.0
703,DIFF_FROM_LINE_betmgm_SEASON_BEFORE_STD_TREND_...,1,0.01,2019.0
663,DIFF_FROM_LINE_fanduel_TREND_SLOPE_LAST_5_HOME...,1,0.01,2020.0


In [49]:
BET365_LINE_COL =  "TOTAL_LINE_bet365"
# BET365_LINE_COL =  "total_bet365_line_over"

# Ensure scoring line and target exist (avoid NaN-driven undefined betting accuracy).
df_to_train = df_to_train.dropna(subset=[BET365_LINE_COL, "TOTAL_POINTS"]).copy()

In [50]:
df_to_train['GAME_DATE'] = pd.to_datetime(df_to_train['GAME_DATE'])
df_to_train = df_to_train.sort_values("GAME_DATE").reset_index(drop=True)

In [51]:
#count games per season
games_per_season = df_to_train.groupby('SEASON_YEAR').size()
print(games_per_season)

SEASON_YEAR
2019     348
2020    1074
2021    1226
2022    1224
2023    1173
2024    1236
2025    1111
dtype: int64


## Train / Test

In [52]:
TARGET_COL = "TOTAL_POINTS"
SAMPLE_WEIGHT_LAMBDA = 0.0075
SAMPLE_WEIGHT_LAMBDA_BOUNDS = (1e-4, 0.015)
TRAIN_GAMES = 7000
DAY_BY_DAY_METRIC_NAME = "OU_Betting_Accuracy"
DAY_BY_DAY_THRESHOLDS = (1, 2, 3)
BET_STAKE_EUR = 1.0
BET_DECIMAL_ODDS = 1.90
BET_WIN_NET_PROFIT_EUR = BET_STAKE_EUR * (BET_DECIMAL_ODDS - 1.0)


In [53]:
df_dev, df_test_final = split_latest_dates_holdout(
    df=df_to_train,
    date_col="GAME_DATE",
    test_size=0.025,
)

print(f"Development set size: {len(df_dev)}")
print(f"Final test set size: {len(df_test_final)}")
print("Final test date range:",
      df_test_final["GAME_DATE"].min(), "->", df_test_final["GAME_DATE"].max())

Development set size: 7203
Final test set size: 189
Final test date range: 2026-03-15 00:00:00 -> 2026-04-08 00:00:00


In [54]:
EXCLUDE_COLS = [
    "TOTAL_POINTS",
    "SEASON_YEAR",
    "GAME_DATE",
]

X_dev = df_dev.drop(columns=EXCLUDE_COLS, errors="ignore")
y_dev = pd.to_numeric(df_dev[TARGET_COL], errors="coerce")
sample_weight_dev = build_recency_sample_weights(
    df_dev,
    lambda_=SAMPLE_WEIGHT_LAMBDA,
)

X_test_final = df_test_final.drop(columns=EXCLUDE_COLS, errors="ignore")
y_test_final = pd.to_numeric(df_test_final[TARGET_COL], errors="coerce")

print(f"X_dev shape: {X_dev.shape}")
print(f"X_test_final shape: {X_test_final.shape}")
print(
    f"Recency sample weights lambda={SAMPLE_WEIGHT_LAMBDA}: "
    f"min={sample_weight_dev.min():.4f}, max={sample_weight_dev.max():.4f}"
)


X_dev shape: (7203, 1370)
X_test_final shape: (189, 1370)
Recency sample weights lambda=0.0075: min=0.0000, max=1.0000


In [55]:
from nba_ou.modeling.scorers import (
    OverUnderScorerTotalPoints,
    OverUnderScorerTotalPointsMinEdge,
    evaluate_total_points_thresholds,
    over_under_betting_accuracy_total_points,
    over_under_betting_accuracy_total_points_with_min_edge,
)

ou_scorer = OverUnderScorerTotalPoints(BET365_LINE_COL)
ou_scorer_edge_2 = OverUnderScorerTotalPointsMinEdge(
    line_col=BET365_LINE_COL,
    min_edge=2,
)
ou_scorer_edge_4 = OverUnderScorerTotalPointsMinEdge(
    line_col=BET365_LINE_COL,
    min_edge=4,
)

scoring = {
    "MAE": "neg_mean_absolute_error",
    "RMSE": "neg_root_mean_squared_error",
    "R2": "r2",
    "OU_Betting_Accuracy": ou_scorer,
    "OU_Betting_Accuracy_Edge_2": ou_scorer_edge_2,
    "OU_Betting_Accuracy_Edge_4": ou_scorer_edge_4,
}


def print_metrics(cv_results):
    for sc in scoring.keys():
        train_key = f"train_{sc}"
        test_key = f"test_{sc}"

        train_vals = cv_results[train_key]
        test_vals = cv_results[test_key]

        train_val = np.nanmean(train_vals)
        test_val = np.nanmean(test_vals)

        if sc in {"MSE", "RMSE", "MAE"}:
            train_val = -train_val
            test_val = -test_val

        if "OU_Betting_Accuracy" in sc:
            print(f"Train {sc}: {train_val:.2%}")
            print(f"Validation {sc}: {test_val:.2%}")
            n_valid = np.sum(~np.isnan(test_vals))
            print(f"  (valid folds: {n_valid}/{len(test_vals)})")
        else:
            print(f"Train {sc}: {train_val:.5f}")
            print(f"Validation {sc}: {test_val:.5f}")
        print()


def calculate_flat_bet_profit_total_points(
    y_true,
    y_pred,
    betting_line,
    *,
    stake=BET_STAKE_EUR,
    win_net_profit=BET_WIN_NET_PROFIT_EUR,
):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    betting_line = np.asarray(betting_line, dtype=float)

    valid = np.isfinite(y_true) & np.isfinite(y_pred) & np.isfinite(betting_line)
    true_side = np.sign(y_true - betting_line)
    pred_side = np.sign(y_pred - betting_line)
    bet_mask = valid & (pred_side != 0)

    profits = np.zeros(len(y_true), dtype=float)
    wins = bet_mask & (true_side == pred_side)
    losses = bet_mask & (true_side != 0) & (true_side != pred_side)
    pushes = bet_mask & (true_side == 0)

    profits[wins] = win_net_profit
    profits[losses] = -stake
    profits[pushes] = 0.0

    return {
        "n_games": int(valid.sum()),
        "n_bets": int(bet_mask.sum()),
        "n_wins": int(wins.sum()),
        "n_losses": int(losses.sum()),
        "n_pushes": int(pushes.sum()),
        "total_profit_eur": float(profits.sum()),
        "avg_profit_per_game_eur": float(profits.sum() / valid.sum()) if valid.sum() else np.nan,
        "avg_profit_per_bet_eur": float(profits.sum() / bet_mask.sum()) if bet_mask.sum() else np.nan,
        "roi_per_bet": float(profits.sum() / (stake * bet_mask.sum())) if bet_mask.sum() else np.nan,
    }


def summarize_walk_forward_total_points(
    predictions_df,
    daily_template,
    df_test_final,
    *,
    line_col=BET365_LINE_COL,
    metric_name=DAY_BY_DAY_METRIC_NAME,
    thresholds=DAY_BY_DAY_THRESHOLDS,
):
    line_lookup = (
        df_test_final.reset_index(drop=True)[[line_col]]
        .reset_index()
        .rename(columns={"index": "row_in_test_final"})
    )

    scored_predictions = predictions_df.merge(
        line_lookup,
        on="row_in_test_final",
        how="left",
        validate="many_to_one",
    ).copy()

    scored_predictions["date"] = pd.to_datetime(
        scored_predictions["date"],
        errors="coerce",
    ).dt.normalize()
    scored_predictions["y_true"] = pd.to_numeric(
        scored_predictions["y_true"],
        errors="coerce",
    )
    scored_predictions["y_pred"] = pd.to_numeric(
        scored_predictions["y_pred"],
        errors="coerce",
    )
    scored_predictions[line_col] = pd.to_numeric(
        scored_predictions[line_col],
        errors="coerce",
    )

    daily_context = daily_template.copy()
    daily_context["date"] = pd.to_datetime(
        daily_context["date"],
        errors="coerce",
    ).dt.normalize()
    daily_context = daily_context.drop(columns=["_walk_mae"], errors="ignore")

    daily_rows = []
    for current_day, day_df in scored_predictions.groupby("date", sort=True):
        context_row = daily_context.loc[daily_context["date"] == current_day].iloc[0].to_dict()
        y_true_day = day_df["y_true"].to_numpy(dtype=float)
        y_pred_day = day_df["y_pred"].to_numpy(dtype=float)
        betting_line_day = day_df[line_col].to_numpy(dtype=float)
        profit_day = calculate_flat_bet_profit_total_points(
            y_true=y_true_day,
            y_pred=y_pred_day,
            betting_line=betting_line_day,
        )

        context_row[metric_name] = over_under_betting_accuracy_total_points(
            y_true=y_true_day,
            y_pred=y_pred_day,
            betting_line=betting_line_day,
        )
        context_row["bet_profit_eur"] = profit_day["total_profit_eur"]
        context_row["avg_profit_per_game_eur"] = profit_day["avg_profit_per_game_eur"]
        context_row["avg_profit_per_bet_eur"] = profit_day["avg_profit_per_bet_eur"]
        context_row["bet_roi"] = profit_day["roi_per_bet"]
        context_row["n_bets"] = profit_day["n_bets"]
        context_row["n_wins"] = profit_day["n_wins"]
        context_row["n_losses"] = profit_day["n_losses"]
        context_row["n_pushes"] = profit_day["n_pushes"]
        daily_rows.append(context_row)

    daily_results = pd.DataFrame(daily_rows)

    y_true = scored_predictions["y_true"].to_numpy(dtype=float)
    y_pred = scored_predictions["y_pred"].to_numpy(dtype=float)
    betting_line = scored_predictions[line_col].to_numpy(dtype=float)
    pred_edge = y_pred - betting_line
    margin = np.abs(pred_edge)
    n_total = len(scored_predictions)
    overall_profit = calculate_flat_bet_profit_total_points(
        y_true=y_true,
        y_pred=y_pred,
        betting_line=betting_line,
    )

    threshold_rows = []
    for threshold in thresholds:
        mask = margin > threshold
        n_games = int(mask.sum())
        threshold_profit = calculate_flat_bet_profit_total_points(
            y_true=y_true[mask],
            y_pred=y_pred[mask],
            betting_line=betting_line[mask],
        )
        ou_acc = (
            np.nan
            if n_games == 0
            else over_under_betting_accuracy_total_points(
                y_true=y_true[mask],
                y_pred=y_pred[mask],
                betting_line=betting_line[mask],
            )
        )
        threshold_rows.append(
            {
                "threshold_abs_pred_edge_gt": threshold,
                "n_games": n_games,
                "pct_of_test": (n_games / n_total) if n_total else np.nan,
                "ou_betting_accuracy": ou_acc,
                "bet_profit_eur": threshold_profit["total_profit_eur"],
                "avg_profit_per_game_eur": threshold_profit["avg_profit_per_game_eur"],
                "avg_profit_per_bet_eur": threshold_profit["avg_profit_per_bet_eur"],
                "bet_roi": threshold_profit["roi_per_bet"],
                "n_bets": threshold_profit["n_bets"],
            }
        )

    threshold_results = pd.DataFrame(threshold_rows)
    return scored_predictions, daily_results, threshold_results, overall_profit


def run_day_by_day_walk_forward_evaluation(
    *,
    label,
    df_dev,
    df_test_final,
    fit_and_predict,
    max_games=TRAIN_GAMES,
    metric_name=DAY_BY_DAY_METRIC_NAME,
    thresholds=DAY_BY_DAY_THRESHOLDS,
):
    raw_result = evaluate_day_by_day_walk_forward(
        df_dev=df_dev,
        df_test_final=df_test_final,
        fit_and_predict=fit_and_predict,
        metric_fn=lambda y_true, y_pred: mean_absolute_error(y_true, y_pred),
        target_col=TARGET_COL,
        max_games=max_games,
        metric_name="_walk_mae",
    )

    scored_predictions, daily_results, threshold_results, overall_profit = summarize_walk_forward_total_points(
        predictions_df=raw_result.predictions,
        daily_template=raw_result.daily_results,
        df_test_final=df_test_final,
        line_col=BET365_LINE_COL,
        metric_name=metric_name,
        thresholds=thresholds,
    )

    mean_metric = float(daily_results[metric_name].mean())
    print(f"{label} mean day-by-day {metric_name}: {mean_metric:.2%}")
    print(
        f"{label} flat-bet profit at {BET_DECIMAL_ODDS:.2f} odds: "
        f"{overall_profit['total_profit_eur']:.2f} EUR total, "
        f"{overall_profit['avg_profit_per_game_eur']:.3f} EUR/game, "
        f"ROI {overall_profit['roi_per_bet']:.2%} over {overall_profit['n_bets']} bets"
    )
    display(
        daily_results.style.format(
            {
                metric_name: "{:.2%}",
                "bet_profit_eur": "{:.2f}",
                "avg_profit_per_game_eur": "{:.3f}",
                "avg_profit_per_bet_eur": "{:.3f}",
                "bet_roi": "{:.2%}",
            }
        )
    )
    print(f"{label} thresholded walk-forward accuracy")
    display(
        threshold_results.style.format(
            {
                "pct_of_test": "{:.1%}",
                "ou_betting_accuracy": "{:.2%}",
                "bet_profit_eur": "{:.2f}",
                "avg_profit_per_game_eur": "{:.3f}",
                "avg_profit_per_bet_eur": "{:.3f}",
                "bet_roi": "{:.2%}",
            }
        )
    )
    return raw_result, scored_predictions, daily_results, threshold_results, overall_profit


In [56]:
splits, fold_info = make_test_anchored_walk_forward_splits(
    df=df_dev,
    date_col="GAME_DATE",
    season_col="SEASON_YEAR",
    test_games=30,
    step_games_between_tests=30,
    train_games=TRAIN_GAMES,
    min_train_games=int(TRAIN_GAMES * 0.75),
    max_folds=15,
    verbose=1,
)

assert_valid_time_splits(df_dev, splits)


Created 15 test-anchored walk-forward folds
 fold  train_n_games  test_n_games train_start_date train_end_date test_start_date test_end_date  test_season
    1           6189            31       2019-11-02     2025-04-11      2025-04-13    2025-04-25         2024
    2           6281            32       2019-11-02     2025-06-22      2025-10-27    2025-11-03         2025
    3           6350            34       2019-11-02     2025-11-08      2025-11-09    2025-11-12         2025
    4           6417            35       2019-11-02     2025-11-17      2025-11-18    2025-11-22         2025
    5           6482            36       2019-11-02     2025-11-26      2025-11-28    2025-12-01         2025
    6           6550            32       2019-11-02     2025-12-05      2025-12-06    2025-12-12         2025
    7           6616            37       2019-11-02     2025-12-19      2025-12-20    2025-12-23         2025
    8           6693            36       2019-11-02     2025-12-29      2025

In [57]:
season_bl = DummyRegressor(strategy="mean")

cv_results = cross_validate(
    season_bl,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("DummyRegressor baseline")
print_metrics(cv_results)

DummyRegressor baseline
Train MAE: 15.55157
Validation MAE: 15.75163

Train RMSE: 19.51192
Validation RMSE: 19.63087

Train R2: 0.00000
Validation R2: -0.11991

Train OU_Betting_Accuracy: 50.38%
Validation OU_Betting_Accuracy: 49.70%
  (valid folds: 15/15)

Train OU_Betting_Accuracy_Edge_2: 50.26%
Validation OU_Betting_Accuracy_Edge_2: 49.72%
  (valid folds: 15/15)

Train OU_Betting_Accuracy_Edge_4: 50.90%
Validation OU_Betting_Accuracy_Edge_4: 49.35%
  (valid folds: 15/15)



In [58]:
lr = LinearRegression()

cv_results = cross_validate(
    lr,
    X_dev.fillna(0),   # LR cannot handle NaNs
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1,
)

print("Linear Regression")
print_metrics(cv_results)

Linear Regression
Train MAE: 12.16728
Validation MAE: 100.55156

Train RMSE: 15.37763
Validation RMSE: 470.92779

Train R2: 0.37885
Validation R2: -4525.72869

Train OU_Betting_Accuracy: 64.15%
Validation OU_Betting_Accuracy: 50.02%
  (valid folds: 15/15)

Train OU_Betting_Accuracy_Edge_2: 67.95%
Validation OU_Betting_Accuracy_Edge_2: 49.83%
  (valid folds: 15/15)

Train OU_Betting_Accuracy_Edge_4: 72.13%
Validation OU_Betting_Accuracy_Edge_4: 50.34%
  (valid folds: 15/15)



In [59]:
xgb_reg_no_weights = XGBRegressor(
    max_depth=3,
    learning_rate=0.05,
    n_estimators=75,
    subsample=0.6,
    colsample_bytree=0.86,
    reg_alpha=0.57,
    reg_lambda=1.78,
    min_child_weight=5.48,
    gamma=1.77,
    n_jobs=-1,
    random_state=16,
)

cv_results_no_weights = cross_validate(
    xgb_reg_no_weights,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("XGBoost no sample weights")
print_metrics(cv_results_no_weights)


XGBoost no sample weights
Train MAE: 12.79502
Validation MAE: 13.56482

Train RMSE: 16.16182
Validation RMSE: 17.10724

Train R2: 0.31391
Validation R2: 0.14593

Train OU_Betting_Accuracy: 64.68%
Validation OU_Betting_Accuracy: 51.63%
  (valid folds: 15/15)

Train OU_Betting_Accuracy_Edge_2: 75.90%
Validation OU_Betting_Accuracy_Edge_2: 51.99%
  (valid folds: 15/15)

Train OU_Betting_Accuracy_Edge_4: 86.07%
Validation OU_Betting_Accuracy_Edge_4: 47.49%
  (valid folds: 15/15)



In [60]:
xgb_reg_no_weights.fit(X_dev, y_dev)

y_pred_test_total = xgb_reg_no_weights.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_total)
rmse = root_mean_squared_error(y_test_final, y_pred_test_total)
mae = mean_absolute_error(y_test_final, y_pred_test_total)

betting_line = X_test_final[BET365_LINE_COL].to_numpy(dtype=float)

ou_acc = over_under_betting_accuracy_total_points(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
)
ou_acc_edge_2 = over_under_betting_accuracy_total_points_with_min_edge(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
    min_edge=2,
)
ou_acc_edge_4 = over_under_betting_accuracy_total_points_with_min_edge(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
    min_edge=4,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")
print(f"OU_Betting_Accuracy_Edge_2: {ou_acc_edge_2:.2%}")
print(f"OU_Betting_Accuracy_Edge_4: {ou_acc_edge_4:.2%}")


Final test metrics
MSE: 307.16824
RMSE: 17.52622
MAE: 13.70295
OU_Betting_Accuracy: 54.84%
OU_Betting_Accuracy_Edge_2: 50.98%
OU_Betting_Accuracy_Edge_4: 60.00%


In [61]:
results_df, y_pred_test_total = evaluate_total_points_thresholds(
    model=xgb_reg_no_weights,
    X_test=X_test_final,
    y_test_total=y_test_final,
    line_col=BET365_LINE_COL,
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "ou_betting_accuracy": "{:.2%}"}
    )
)


def fit_and_predict_xgb_no_weights_day_by_day(train_df, test_df):
    model = XGBRegressor(**xgb_reg_no_weights.get_params())

    X_train = train_df.drop(columns=EXCLUDE_COLS, errors="ignore")
    y_train = pd.to_numeric(train_df[TARGET_COL], errors="coerce")
    X_test = test_df.drop(columns=EXCLUDE_COLS, errors="ignore")

    model.fit(X_train, y_train)
    return model.predict(X_test)


day_by_day_no_weights, day_by_day_no_weights_predictions, day_by_day_no_weights_daily, day_by_day_no_weights_thresholds, day_by_day_no_weights_profit = run_day_by_day_walk_forward_evaluation(
    label="XGBoost no sample weights",
    df_dev=df_dev,
    df_test_final=df_test_final,
    fit_and_predict=fit_and_predict_xgb_no_weights_day_by_day,
    max_games=TRAIN_GAMES,
)

,threshold_abs_pred_edge_gt,n_games,pct_of_test,ou_betting_accuracy
0,0,189,100.0%,54.84%
1,1,103,54.5%,55.45%
2,2,51,27.0%,50.98%
3,3,23,12.2%,52.17%
4,4,10,5.3%,60.00%
5,5,5,2.6%,60.00%
6,6,2,1.1%,50.00%
7,7,1,0.5%,100.00%
8,8,0,0.0%,nan%
9,9,0,0.0%,nan%


XGBoost no sample weights mean day-by-day OU_Betting_Accuracy: 51.67%
XGBoost no sample weights flat-bet profit at 1.90 odds: -3.60 EUR total, -0.019 EUR/game, ROI -1.90% over 189 bets


,date,train_n_games,test_n_games,train_start_date,train_end_date,OU_Betting_Accuracy,bet_profit_eur,avg_profit_per_game_eur,avg_profit_per_bet_eur,bet_roi,n_bets,n_wins,n_losses,n_pushes
0,2026-03-15 00:00:00,7000,7,2020-02-28 00:00:00,2026-03-14 00:00:00,42.86%,-1.30,-0.186,-0.186,-18.57%,7,3,4,0
1,2026-03-16 00:00:00,7000,8,2020-02-29 00:00:00,2026-03-15 00:00:00,57.14%,0.60,0.075,0.075,7.50%,8,4,3,1
2,2026-03-17 00:00:00,7000,8,2020-03-01 00:00:00,2026-03-16 00:00:00,50.00%,-0.40,-0.050,-0.050,-5.00%,8,4,4,0
3,2026-03-18 00:00:00,7000,9,2020-03-02 00:00:00,2026-03-17 00:00:00,22.22%,-5.20,-0.578,-0.578,-57.78%,9,2,7,0
4,2026-03-19 00:00:00,7000,8,2020-03-03 00:00:00,2026-03-18 00:00:00,37.50%,-2.30,-0.288,-0.288,-28.75%,8,3,5,0
5,2026-03-20 00:00:00,7000,6,2020-03-04 00:00:00,2026-03-19 00:00:00,50.00%,-0.30,-0.050,-0.050,-5.00%,6,3,3,0
6,2026-03-21 00:00:00,7000,10,2020-03-05 00:00:00,2026-03-20 00:00:00,50.00%,-0.50,-0.050,-0.050,-5.00%,10,5,5,0
7,2026-03-22 00:00:00,7000,5,2020-03-06 00:00:00,2026-03-21 00:00:00,60.00%,0.70,0.140,0.140,14.00%,5,3,2,0
8,2026-03-23 00:00:00,7000,10,2020-03-07 00:00:00,2026-03-22 00:00:00,70.00%,3.30,0.330,0.330,33.00%,10,7,3,0
9,2026-03-24 00:00:00,7000,4,2020-03-08 00:00:00,2026-03-23 00:00:00,50.00%,-0.20,-0.050,-0.050,-5.00%,4,2,2,0


XGBoost no sample weights thresholded walk-forward accuracy


,threshold_abs_pred_edge_gt,n_games,pct_of_test,ou_betting_accuracy,bet_profit_eur,avg_profit_per_game_eur,avg_profit_per_bet_eur,bet_roi,n_bets
0,1,108,57.1%,55.14%,5.10,0.047,0.047,4.72%,108
1,2,51,27.0%,62.75%,9.80,0.192,0.192,19.22%,51
2,3,21,11.1%,47.62%,-2.00,-0.095,-0.095,-9.52%,21


In [62]:
xgb_reg_weights = XGBRegressor(
    max_depth=3,
    learning_rate=0.05,
    n_estimators=75,
    subsample=0.6,
    colsample_bytree=0.86,
    reg_alpha=0.57,
    reg_lambda=1.78,
    min_child_weight=5.48,
    gamma=1.77,
    n_jobs=-1,
    random_state=16,
)

weighted_xgb = TemporalDecaySampleWeightRegressor(
    estimator=xgb_reg_weights,
    dates=df_dev["GAME_DATE"],
    lambda_=SAMPLE_WEIGHT_LAMBDA,
)

cv_results_weights = cross_validate(
    weighted_xgb,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("XGBoost with sample weights (per-fold decay)")
print_metrics(cv_results_weights)


XGBoost with sample weights (per-fold decay)
Train MAE: 13.51284
Validation MAE: 13.63134

Train RMSE: 17.05520
Validation RMSE: 17.22782

Train R2: 0.23589
Validation R2: 0.13516

Train OU_Betting_Accuracy: 55.36%
Validation OU_Betting_Accuracy: 52.69%
  (valid folds: 15/15)

Train OU_Betting_Accuracy_Edge_2: 57.75%
Validation OU_Betting_Accuracy_Edge_2: 51.47%
  (valid folds: 15/15)

Train OU_Betting_Accuracy_Edge_4: 60.76%
Validation OU_Betting_Accuracy_Edge_4: 50.78%
  (valid folds: 15/15)



In [63]:
weighted_xgb.fit(X_dev, y_dev)

y_pred_test_total = weighted_xgb.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_total)
rmse = root_mean_squared_error(y_test_final, y_pred_test_total)
mae = mean_absolute_error(y_test_final, y_pred_test_total)

betting_line = X_test_final[BET365_LINE_COL].to_numpy(dtype=float)

ou_acc = over_under_betting_accuracy_total_points(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
)
ou_acc_edge_2 = over_under_betting_accuracy_total_points_with_min_edge(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
    min_edge=2,
)
ou_acc_edge_4 = over_under_betting_accuracy_total_points_with_min_edge(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
    min_edge=4,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")
print(f"OU_Betting_Accuracy_Edge_2: {ou_acc_edge_2:.2%}")
print(f"OU_Betting_Accuracy_Edge_4: {ou_acc_edge_4:.2%}")


Final test metrics
MSE: 330.50482
RMSE: 18.17979
MAE: 14.33557
OU_Betting_Accuracy: 46.24%
OU_Betting_Accuracy_Edge_2: 44.95%
OU_Betting_Accuracy_Edge_4: 46.15%


In [64]:
results_df, y_pred_test_total = evaluate_total_points_thresholds(
    model=weighted_xgb,
    X_test=X_test_final,
    y_test_total=y_test_final,
    line_col=BET365_LINE_COL,
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "ou_betting_accuracy": "{:.2%}"}
    )
)


def fit_and_predict_xgb_weights_day_by_day(train_df, test_df):
    base_model = XGBRegressor(**xgb_reg_weights.get_params())
    model = TemporalDecaySampleWeightRegressor(
        estimator=base_model,
        dates=train_df["GAME_DATE"],
        lambda_=SAMPLE_WEIGHT_LAMBDA,
    )

    X_train = train_df.drop(columns=EXCLUDE_COLS, errors="ignore")
    y_train = pd.to_numeric(train_df[TARGET_COL], errors="coerce")
    X_test = test_df.drop(columns=EXCLUDE_COLS, errors="ignore")

    model.fit(X_train, y_train)
    return model.predict(X_test)


day_by_day_weights, day_by_day_weights_predictions, day_by_day_weights_daily, day_by_day_weights_thresholds, day_by_day_weights_profit = run_day_by_day_walk_forward_evaluation(
    label="XGBoost with sample weights",
    df_dev=df_dev,
    df_test_final=df_test_final,
    fit_and_predict=fit_and_predict_xgb_weights_day_by_day,
    max_games=TRAIN_GAMES,
)


,threshold_abs_pred_edge_gt,n_games,pct_of_test,ou_betting_accuracy
0,0,189,100.0%,46.24%
1,1,145,76.7%,45.77%
2,2,111,58.7%,44.95%
3,3,81,42.9%,47.50%
4,4,52,27.5%,46.15%
5,5,39,20.6%,46.15%
6,6,26,13.8%,42.31%
7,7,20,10.6%,50.00%
8,8,10,5.3%,40.00%
9,9,5,2.6%,0.00%


XGBoost with sample weights mean day-by-day OU_Betting_Accuracy: 48.80%
XGBoost with sample weights flat-bet profit at 1.90 odds: -7.40 EUR total, -0.039 EUR/game, ROI -3.92% over 189 bets


,date,train_n_games,test_n_games,train_start_date,train_end_date,OU_Betting_Accuracy,bet_profit_eur,avg_profit_per_game_eur,avg_profit_per_bet_eur,bet_roi,n_bets,n_wins,n_losses,n_pushes
0,2026-03-15 00:00:00,7000,7,2020-02-28 00:00:00,2026-03-14 00:00:00,57.14%,0.60,0.086,0.086,8.57%,7,4,3,0
1,2026-03-16 00:00:00,7000,8,2020-02-29 00:00:00,2026-03-15 00:00:00,57.14%,0.60,0.075,0.075,7.50%,8,4,3,1
2,2026-03-17 00:00:00,7000,8,2020-03-01 00:00:00,2026-03-16 00:00:00,37.50%,-2.30,-0.288,-0.288,-28.75%,8,3,5,0
3,2026-03-18 00:00:00,7000,9,2020-03-02 00:00:00,2026-03-17 00:00:00,22.22%,-5.20,-0.578,-0.578,-57.78%,9,2,7,0
4,2026-03-19 00:00:00,7000,8,2020-03-03 00:00:00,2026-03-18 00:00:00,50.00%,-0.40,-0.050,-0.050,-5.00%,8,4,4,0
5,2026-03-20 00:00:00,7000,6,2020-03-04 00:00:00,2026-03-19 00:00:00,83.33%,3.50,0.583,0.583,58.33%,6,5,1,0
6,2026-03-21 00:00:00,7000,10,2020-03-05 00:00:00,2026-03-20 00:00:00,40.00%,-2.40,-0.240,-0.240,-24.00%,10,4,6,0
7,2026-03-22 00:00:00,7000,5,2020-03-06 00:00:00,2026-03-21 00:00:00,20.00%,-3.10,-0.620,-0.620,-62.00%,5,1,4,0
8,2026-03-23 00:00:00,7000,10,2020-03-07 00:00:00,2026-03-22 00:00:00,60.00%,1.40,0.140,0.140,14.00%,10,6,4,0
9,2026-03-24 00:00:00,7000,4,2020-03-08 00:00:00,2026-03-23 00:00:00,25.00%,-2.10,-0.525,-0.525,-52.50%,4,1,3,0


XGBoost with sample weights thresholded walk-forward accuracy


,threshold_abs_pred_edge_gt,n_games,pct_of_test,ou_betting_accuracy,bet_profit_eur,avg_profit_per_game_eur,avg_profit_per_bet_eur,bet_roi,n_bets
0,1,148,78.3%,46.58%,-16.80,-0.114,-0.114,-11.35%,148
1,2,113,59.8%,45.95%,-14.10,-0.125,-0.125,-12.48%,113
2,3,78,41.3%,42.11%,-15.20,-0.195,-0.195,-19.49%,78


# OPTUNA

In [65]:
from nba_ou.modeling.optuna_total_points import (
    fit_best_xgb_total_points,
    select_best_trial_lexicographic,
    summarize_lexicographic_candidates,
    summarize_optuna_trials,
    tune_xgb_total_points_optuna,
)

study = tune_xgb_total_points_optuna(
    X=X_dev,
    y=y_dev,
    sample_weight_dates=df_dev["GAME_DATE"],
    tune_sample_weight_lambda=True,
    sample_weight_lambda_bounds=SAMPLE_WEIGHT_LAMBDA_BOUNDS,
    splits=splits,
    line_col=BET365_LINE_COL,
    n_trials=80,
    timeout=4.5 * 3600,
    objective_name="reg:squarederror",
    study_name="xgb_total_points_mae",
)

best_trial_lexi = select_best_trial_lexicographic(
    study,
    mae_tolerance_abs=0.05,
)

print("Optuna best by MAE only")
print("Trial:", study.best_trial.number)
print("Best CV MAE:", study.best_value)
print("Mean OU accuracy:", study.best_trial.user_attrs.get("mean_ou_acc"))
print("Mean OU accuracy edge 2:", study.best_trial.user_attrs.get("mean_ou_acc_edge_2"))
print("Mean OU accuracy edge 4:", study.best_trial.user_attrs.get("mean_ou_acc_edge_4"))
print("Sample weight lambda:", study.best_trial.user_attrs.get("sample_weight_lambda"))

print()
print("Selected trial after MAE-first / OU-second ranking")
print("Trial:", best_trial_lexi.number)
print("CV MAE:", best_trial_lexi.user_attrs.get("mean_mae", best_trial_lexi.value))
print("Mean RMSE:", best_trial_lexi.user_attrs.get("mean_rmse"))
print("Mean R2:", best_trial_lexi.user_attrs.get("mean_r2"))
print("Mean OU accuracy:", best_trial_lexi.user_attrs.get("mean_ou_acc"))
print("Mean OU accuracy edge 2:", best_trial_lexi.user_attrs.get("mean_ou_acc_edge_2"))
print("Mean OU accuracy edge 4:", best_trial_lexi.user_attrs.get("mean_ou_acc_edge_4"))
print("Median best_iteration:", best_trial_lexi.user_attrs.get("median_best_iteration"))
print("Sample weight lambda:", best_trial_lexi.params.get("sample_weight_lambda"))
print("Params:")
for k, v in best_trial_lexi.params.items():
    print(f"{k}: {v}")

trials_df = summarize_optuna_trials(study)
display(
    trials_df.head(15).style.format(
        {
            "value_mae": "{:.4f}",
            "mean_rmse": "{:.4f}",
            "mean_r2": "{:.4f}",
            "mean_ou_acc": "{:.2%}",
            "mean_ou_acc_edge_2": "{:.2%}",
            "mean_ou_acc_edge_3": "{:.2%}",
            "mean_ou_acc_edge_4": "{:.2%}",
        }
    )
)

candidates_df = summarize_lexicographic_candidates(
    study,
    mae_tolerance_abs=0.05,
)

display(
    candidates_df.head(15).style.format(
        {
            "value_mae": "{:.4f}",
            "mean_mae": "{:.4f}",
            "mean_rmse": "{:.4f}",
            "mean_r2": "{:.4f}",
            "mean_ou_acc": "{:.2%}",
            "mean_ou_acc_edge_2": "{:.2%}",
            "mean_ou_acc_edge_3": "{:.2%}",
            "mean_ou_acc_edge_4": "{:.2%}",
        }
    )
)


[I 2026-04-11 18:14:11,762] A new study created in memory with name: xgb_total_points_mae


  0%|          | 0/80 [00:00<?, ?it/s]

[I 2026-04-11 18:27:31,800] Trial 0 finished with value: 13.315471301512337 and parameters: {'max_depth': 2, 'min_child_weight': 18.346704707583235, 'gamma': 1.6970342240854253, 'subsample': 0.5682407800531226, 'colsample_bytree': 0.5123279759064977, 'learning_rate': 0.011926786034588454, 'reg_alpha': 1.8771791376898666, 'reg_lambda': 1.897469395521307, 'sample_weight_lambda': 0.0001422437941448231}. Best is trial 0 with value: 13.315471301512337.
[I 2026-04-11 18:45:13,787] Trial 1 finished with value: 13.2332312222829 and parameters: {'max_depth': 4, 'min_child_weight': 20.290108931287893, 'gamma': 0.3261777842309625, 'subsample': 0.8390562044499359, 'colsample_bytree': 0.4213034780854249, 'learning_rate': 0.012620826760486504, 'reg_alpha': 0.09307011182812809, 'reg_lambda': 15.258811505244246, 'sample_weight_lambda': 0.0010239553604139541}. Best is trial 1 with value: 13.2332312222829.
[I 2026-04-11 18:54:00,565] Trial 2 finished with value: 13.265452275675422 and parameters: {'max_

,trial,value_mae,mean_rmse,mean_r2,mean_ou_acc,mean_ou_acc_edge_2,mean_ou_acc_edge_3,mean_ou_acc_edge_4,mean_best_iteration,median_best_iteration,max_depth,min_child_weight,gamma,subsample,colsample_bytree,learning_rate,reg_alpha,reg_lambda,sample_weight_lambda
0,29,13.0101,16.6510,0.1915,58.93%,61.34%,60.12%,55.75%,193,197,2,17.539262,1.467045,0.593414,0.529664,0.042281,0.079565,3.630081,0.001538
1,21,13.0451,16.7406,0.1822,59.01%,61.68%,59.33%,58.74%,95,56,3,26.354090,2.561570,0.615573,0.465358,0.057206,0.028534,7.086939,0.004023
2,30,13.0941,16.7195,0.1854,58.70%,57.88%,59.62%,55.30%,201,170,2,18.284831,1.515837,0.591831,0.524134,0.042402,0.072901,3.569167,0.000386
3,4,13.1073,16.7965,0.1783,58.50%,61.64%,60.53%,62.66%,141,95,3,27.769564,0.863656,0.554592,0.697083,0.040937,0.028904,9.594362,0.000551
4,16,13.1118,16.8802,0.1690,55.00%,57.70%,58.48%,58.25%,150,113,3,5.708433,1.946339,0.741626,0.608013,0.054939,0.237860,27.084743,0.001919
5,13,13.1291,16.7776,0.1800,58.84%,59.37%,55.36%,55.09%,91,60,3,38.758505,2.273138,0.641540,0.477708,0.059463,0.034188,2.718575,0.002564
6,14,13.1312,16.9002,0.1675,56.57%,58.56%,59.30%,58.33%,110,80,3,38.312980,2.198923,0.623297,0.363375,0.059010,0.041729,2.523795,0.002742
7,12,13.1431,16.7216,0.1848,58.14%,63.67%,62.36%,70.49%,197,128,3,56.378638,2.130883,0.662731,0.500920,0.029101,0.015497,3.218398,0.000630
8,11,13.1591,16.7921,0.1787,57.75%,61.76%,61.31%,64.86%,207,217,3,57.111395,2.831450,0.687739,0.539153,0.030290,0.021864,4.817041,0.000499
9,22,13.1612,16.8565,0.1714,56.15%,57.44%,58.92%,60.78%,143,99,3,26.977384,2.647291,0.598382,0.645559,0.037370,0.029381,15.505458,0.004577


,trial,value_mae,mean_mae,mean_rmse,mean_r2,mean_ou_acc,mean_ou_acc_edge_2,mean_ou_acc_edge_3,mean_ou_acc_edge_4,mean_best_iteration,median_best_iteration,mae_cutoff,max_depth,min_child_weight,gamma,subsample,colsample_bytree,learning_rate,reg_alpha,reg_lambda,sample_weight_lambda
0,21,13.0451,13.0451,16.7406,0.1822,59.01%,61.68%,59.33%,58.74%,95,56,13.060101,3,26.354090,2.561570,0.615573,0.465358,0.057206,0.028534,7.086939,0.004023
1,29,13.0101,13.0101,16.6510,0.1915,58.93%,61.34%,60.12%,55.75%,193,197,13.060101,2,17.539262,1.467045,0.593414,0.529664,0.042281,0.079565,3.630081,0.001538


In [66]:
def fit_and_predict_optuna_day_by_day(train_df, test_df):
    X_train = train_df.drop(columns=EXCLUDE_COLS, errors="ignore")
    y_train = pd.to_numeric(train_df[TARGET_COL], errors="coerce")
    X_test = test_df.drop(columns=EXCLUDE_COLS, errors="ignore")

    model = fit_best_xgb_total_points(
        X_dev=X_train,
        y_dev=y_train,
        sample_weight_dates=train_df["GAME_DATE"],
        sample_weight_lambda=best_trial_lexi.params.get("sample_weight_lambda"),
        trial=best_trial_lexi,
        objective_name="reg:squarederror",
    )
    return model.predict(X_test)


day_by_day_optuna, day_by_day_optuna_predictions, day_by_day_optuna_daily, day_by_day_optuna_thresholds, day_by_day_optuna_profit = run_day_by_day_walk_forward_evaluation(
    label="Optuna-selected XGBoost",
    df_dev=df_dev,
    df_test_final=df_test_final,
    fit_and_predict=fit_and_predict_optuna_day_by_day,
    max_games=TRAIN_GAMES,
)

total_df = df_dev.tail(TRAIN_GAMES)


Optuna-selected XGBoost mean day-by-day OU_Betting_Accuracy: 45.42%
Optuna-selected XGBoost flat-bet profit at 1.90 odds: -26.40 EUR total, -0.140 EUR/game, ROI -13.97% over 189 bets


,date,train_n_games,test_n_games,train_start_date,train_end_date,OU_Betting_Accuracy,bet_profit_eur,avg_profit_per_game_eur,avg_profit_per_bet_eur,bet_roi,n_bets,n_wins,n_losses,n_pushes
0,2026-03-15 00:00:00,7000,7,2020-02-28 00:00:00,2026-03-14 00:00:00,71.43%,2.50,0.357,0.357,35.71%,7,5,2,0
1,2026-03-16 00:00:00,7000,8,2020-02-29 00:00:00,2026-03-15 00:00:00,42.86%,-1.30,-0.163,-0.163,-16.25%,8,3,4,1
2,2026-03-17 00:00:00,7000,8,2020-03-01 00:00:00,2026-03-16 00:00:00,12.50%,-6.10,-0.762,-0.762,-76.25%,8,1,7,0
3,2026-03-18 00:00:00,7000,9,2020-03-02 00:00:00,2026-03-17 00:00:00,22.22%,-5.20,-0.578,-0.578,-57.78%,9,2,7,0
4,2026-03-19 00:00:00,7000,8,2020-03-03 00:00:00,2026-03-18 00:00:00,37.50%,-2.30,-0.288,-0.288,-28.75%,8,3,5,0
5,2026-03-20 00:00:00,7000,6,2020-03-04 00:00:00,2026-03-19 00:00:00,83.33%,3.50,0.583,0.583,58.33%,6,5,1,0
6,2026-03-21 00:00:00,7000,10,2020-03-05 00:00:00,2026-03-20 00:00:00,30.00%,-4.30,-0.430,-0.430,-43.00%,10,3,7,0
7,2026-03-22 00:00:00,7000,5,2020-03-06 00:00:00,2026-03-21 00:00:00,40.00%,-1.20,-0.240,-0.240,-24.00%,5,2,3,0
8,2026-03-23 00:00:00,7000,10,2020-03-07 00:00:00,2026-03-22 00:00:00,50.00%,-0.50,-0.050,-0.050,-5.00%,10,5,5,0
9,2026-03-24 00:00:00,7000,4,2020-03-08 00:00:00,2026-03-23 00:00:00,25.00%,-2.10,-0.525,-0.525,-52.50%,4,1,3,0


Optuna-selected XGBoost thresholded walk-forward accuracy


,threshold_abs_pred_edge_gt,n_games,pct_of_test,ou_betting_accuracy,bet_profit_eur,avg_profit_per_game_eur,avg_profit_per_bet_eur,bet_roi,n_bets
0,1,133,70.4%,43.85%,-21.70,-0.163,-0.163,-16.32%,133
1,2,91,48.1%,43.96%,-15.00,-0.165,-0.165,-16.48%,91
2,3,55,29.1%,43.64%,-9.40,-0.171,-0.171,-17.09%,55


In [67]:
from nba_ou.modeling.modeling import ModelBundleMetadata, ModelInfo, TrainingMetrics

X_dev = total_df.drop(columns=EXCLUDE_COLS, errors="ignore")
y_dev = pd.to_numeric(total_df[TARGET_COL], errors="coerce")
sample_weight_dates_dev = total_df["GAME_DATE"]

best_model = fit_best_xgb_total_points(
    X_dev=X_dev,
    y_dev=y_dev,
    sample_weight_dates=sample_weight_dates_dev,
    sample_weight_lambda=best_trial_lexi.params.get("sample_weight_lambda"),
    trial=best_trial_lexi,
    objective_name="reg:squarederror",
)

y_pred_test_total = best_model.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_total)
rmse = root_mean_squared_error(y_test_final, y_pred_test_total)
mae = mean_absolute_error(y_test_final, y_pred_test_total)

betting_line = X_test_final[BET365_LINE_COL].to_numpy(dtype=float)

ou_acc = over_under_betting_accuracy_total_points(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
)
ou_acc_edge_2 = over_under_betting_accuracy_total_points_with_min_edge(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
    min_edge=2,
)
ou_acc_edge_4 = over_under_betting_accuracy_total_points_with_min_edge(
    y_true=y_test_final,
    y_pred=y_pred_test_total,
    betting_line=betting_line,
    min_edge=4,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")
print(f"OU_Betting_Accuracy_Edge_2: {ou_acc_edge_2:.2%}")
print(f"OU_Betting_Accuracy_Edge_4: {ou_acc_edge_4:.2%}")

results_df, y_pred_test_total = evaluate_total_points_thresholds(
    model=best_model,
    X_test=X_test_final,
    y_test_total=y_test_final,
    line_col=BET365_LINE_COL,
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "ou_betting_accuracy": "{:.2%}"}
    )
)

df_to_train_split_rows = df_to_train.copy().tail(TRAIN_GAMES)

X_full = df_to_train_split_rows.drop(columns=EXCLUDE_COLS, errors="ignore")
y_full = pd.to_numeric(df_to_train_split_rows[TARGET_COL], errors="coerce")
sample_weight_dates_full = df_to_train_split_rows["GAME_DATE"]

production_model = fit_best_xgb_total_points(
    X_dev=X_full,
    y_dev=y_full,
    sample_weight_dates=sample_weight_dates_full,
    sample_weight_lambda=best_trial_lexi.params.get("sample_weight_lambda"),
    trial=best_trial_lexi,
    objective_name="reg:squarederror",
)

latest_training_date = pd.to_datetime(df_to_train_split_rows["GAME_DATE"]).max()
model_version = latest_training_date.strftime("%d_%m_%y")
model_name = f"three_seasons_xgb_total_points_{model_version}"

metadata = ModelBundleMetadata(
    model_info=ModelInfo(
        name=model_name,
        model_version=model_version,
        model_type="three_seasons_total_points",
        prediction_source="three_seasons_xgb_total_points",
        training_code_tag="1.0",
    ),
    training_metrics=TrainingMetrics(
        best_params=best_trial_lexi.params,
        selected_trial_number=best_trial_lexi.number,
        mean_best_iteration=best_trial_lexi.user_attrs.get("mean_best_iteration"),
        median_best_iteration=best_trial_lexi.user_attrs.get("median_best_iteration"),
        cv_mae=float(best_trial_lexi.user_attrs.get("mean_mae", best_trial_lexi.value)),
        cv_rmse=best_trial_lexi.user_attrs.get("mean_rmse"),
        cv_ou_acc=best_trial_lexi.user_attrs.get("mean_ou_acc"),
        final_test_mae=float(mae),
        final_test_rmse=float(rmse),
        final_test_ou_acc=float(ou_acc),
        nan_threshold=nan_threshold,
        max_na_per_row=max_na_per_row,
        train_date_min=df_to_train_split_rows["GAME_DATE"].min().to_pydatetime(),
        train_date_max=df_to_train_split_rows["GAME_DATE"].max().to_pydatetime(),
        train_games=TRAIN_GAMES,
        sample_weight_lambda_bounds=SAMPLE_WEIGHT_LAMBDA_BOUNDS,
    ),
)

model_path, meta_path = save_model_bundle(
    model=production_model,
    feature_names=list(X_full.columns),
    out_dir="/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/all_seasons_weighted/",
    metadata=metadata,
)

print(
    f"Production model trained on {len(X_full)} rows using fixed n_estimators from median_best_iteration."
)
print("Saved model :", model_path)
print("Saved metadata:", meta_path)


Final test metrics
MSE: 330.50583
RMSE: 18.17982
MAE: 14.16044
OU_Betting_Accuracy: 52.69%
OU_Betting_Accuracy_Edge_2: 46.23%
OU_Betting_Accuracy_Edge_4: 58.54%


,threshold_abs_pred_edge_gt,n_games,pct_of_test,ou_betting_accuracy
0,0,189,100.0%,52.69%
1,1,142,75.1%,48.57%
2,2,108,57.1%,46.23%
3,3,74,39.2%,52.05%
4,4,42,22.2%,58.54%
5,5,28,14.8%,51.85%
6,6,16,8.5%,43.75%
7,7,9,4.8%,33.33%
8,8,8,4.2%,37.50%
9,9,4,2.1%,25.00%


Production model trained on 7000 rows using fixed n_estimators from median_best_iteration.
Saved model : /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/all_seasons_weighted/three_seasons_xgb_total_points_08_04_26.json
Saved metadata: /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/total_points/all_seasons_weighted/three_seasons_xgb_total_points_08_04_26.meta.json
